# The best PyNeuraLogic I-RAVEN model — architecture walkthrough

This notebook documents the model currently training as **`experiments/max_data_L2`**: the
`--level 2` all-constellations architecture from
[`train_all_constellations.py`](train_all_constellations.py), trained jointly on all seven
I-RAVEN constellations.

| | |
|---|---|
| template | PyNeuraLogic: 3 message-passing layers **and** the scoring head (~87% of parameters) |
| torch side | attribute embeddings, pad token, input projection, loss, AdamW (~13%) |
| topology | one graph per puzzle: 16 panels × 9 padded slots = **144 object nodes**, 1656 edges |
| dims | embed 128, hidden 128, 3 layers, LeakyReLU, AVG aggregation |
| data | 6000 puzzles per constellation (the full train split), 42,000 total |
| checkpoint | the run's current best, loaded live (§8 prints its epoch); at epoch 62 it scored **64.00% test**, 63.96% from pixels, against 45.3% ± 12.4 for the torch GNN over three seeds |

Nothing here is a re-implementation: the notebook **imports the real training module**, so every
rule and tensor you see below is the one used in the run.

Run it with the `pynl` kernel (conda env `pynl`, PyNeuraLogic 0.9.2, Java 21) on a **compute
node**: the login node kills CPU-heavy Jupyter kernels (SIGKILL). Batch version:

```bash
sbatch --partition=amdfast --cpus-per-task=16 --mem=32G --time=0:30:00 --wrap \
  "source /mnt/appl/software/Anaconda3/2024.02-1/etc/profile.d/conda.sh && conda activate pynl && \
   export PYTHONNOUSERSITE=1 JAVA_HOME=/mnt/appl/software/Java/21.0.8 && \
   jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=python3 \
   pynl_best_architecture_walkthrough.ipynb"
```

All puzzles shown are **unseen validation puzzles** (index 300 on): never trained on, and not
used to pick the checkpoint. The test split is kept for one final evaluation.

In [1]:
import os, sys, glob, shutil, tempfile
import numpy as np
import torch

# A JVM with no ceiling will happily take the whole machine and kill the kernel.
import neuralogic
neuralogic.set_max_memory_size(30)          # GB
from neuralogic.logging import clear_handlers
clear_handlers()                            # silence the Java engine's INFO/GC console log

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")   # paths below are relative to the repo root
ROOT = os.path.abspath(".")
sys.path[:0] = [ROOT, os.path.join(ROOT, "pynl")]

from neuralogic.core import Transformation, Aggregation

# the real training module -- not a copy
import train_all_constellations as T
from train_all_constellations import (NODES, SLOTS, ATTRS, EDGE_TYPES, POS_LIST,
                                      CONSTELLATIONS, edges, structural_facts,
                                      build_template, AllConstellations, encode_puzzle)
from train_pynl import parse, POS_MAP

print("java:", os.environ.get("JAVA_HOME", "<unset>"))
print("neuralogic:", neuralogic.__version__ if hasattr(neuralogic, "__version__") else "0.9.2")
print("torch:", torch.__version__)

java: /mnt/appl/software/Java/21.0.8
neuralogic: 0.9.2
torch: 2.14.0+cpu


/home/kassyask/.conda/envs/pynl/lib/python3.11/site-packages/neuralogic/core/builder/builder.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## 1. The graph the model reasons over

Every I-RAVEN puzzle is 16 panels: 8 context panels (the 3×3 matrix minus its last cell) and
8 candidate answers. Objects sit at **22 universal positions** across the dataset, but each
constellation only uses a few of them:

In [2]:
print(f"{'constellation':<40} {'#positions':>10}   universal position ids")
for cfg in CONSTELLATIONS:
    print(f"{cfg:<40} {len(POS_LIST[cfg]):>10}   {POS_LIST[cfg]}")
print(f"\nSLOTS (max over constellations) = {SLOTS}")

constellation                            #positions   universal position ids
center_single                                     1   [0]
distribute_four                                   4   [1, 2, 3, 4]
distribute_nine                                   9   [5, 6, 7, 8, 9, 10, 11, 12, 13]
left_center_single_right_center_single            2   [14, 15]
up_center_single_down_center_single               2   [16, 17]
in_center_single_out_center_single                2   [0, 9]
in_distribute_four_out_center_single              5   [0, 18, 19, 20, 21]

SLOTS (max over constellations) = 9


### Why the topology is padded

PyNeuraLogic compiles one static computation graph, and template weights cannot be shared
across *different* topologies without breaking the backward pass. So all seven constellations
are forced onto **one common shape**: 9 slots per panel, where

> **slot k = the k-th universal position of this constellation**

A constellation with fewer positions leaves the remaining slots inactive, and every inactive
slot gets a single **learned pad embedding** (`self.absent`) instead of an attribute sum. Slot
identity is therefore consistent within a constellation, which is what the same-slot
cross-panel edges rely on.

In [3]:
print(f"nodes: {len(NODES)}  = 16 panels x {SLOTS} slots")
print("first:", NODES[:4], "...")
print("last :", NODES[-4:])
print("\nnode name pXX_sYY  ->  panel XX (0-7 context, 8-15 candidates), slot YY")

nodes: 144  = 16 panels x 9 slots
first: ['p0_s0', 'p0_s1', 'p0_s2', 'p0_s3'] ...
last : ['p15_s5', 'p15_s6', 'p15_s7', 'p15_s8']

node name pXX_sYY  ->  panel XX (0-7 context, 8-15 candidates), slot YY


## 2. Edges

Three relations carry all the information flow. Every edge is typed, and the type gets its own
learned embedding in the template.

In [4]:
from collections import Counter

for same_panel in (True, False):
    e = edges(same_panel=same_panel)
    print(f"same_panel={str(same_panel):<5} total {len(e):>5}   {dict(Counter(r for _, _, r in e))}")

print("\nexamples (the config actually used: --no-same-panel):")
for s, t, r in edges(same_panel=False)[:3]:
    print(f"   {s:>7} -> {t:<7}  {r}")

same_panel=True  total  2808   {'same_panel': 1152, 'ctx_ctx': 504, 'ctx_cand': 576, 'cand_ctx': 576}
same_panel=False total  1656   {'ctx_ctx': 504, 'ctx_cand': 576, 'cand_ctx': 576}

examples (the config actually used: --no-same-panel):
     p0_s0 -> p1_s0    ctx_ctx
     p0_s0 -> p2_s0    ctx_ctx
     p0_s0 -> p3_s0    ctx_ctx


`ctx_ctx` links the same slot across the 8 context panels, `ctx_cand` and `cand_ctx` link
context slots to candidate slots in both directions. **Candidates are never connected to each
other** — each one is scored against the context alone, exactly as the torch GNN does with its
8 separate graphs.

The `same_panel` all-pairs edges (within one panel) are **dropped** in the best config: a grid
search found they hurt, mostly because on padded constellations they are overwhelmingly
pad-to-pad edges. That cuts the graph from 2808 to 1656 edges.

## 3. One real puzzle → tensors

`parse()` reads the ground truth objects out of the XML (positions decoded geometrically — see
the `canonical_positions` fix in `gnn/data.py`), and `encode_puzzle()` lays them into the
padded slots.

In [5]:
CFG = "distribute_nine"          # 9 slots: the constellation that sets SLOTS
f = sorted(glob.glob(f"data/I-RAVEN-10000/{CFG}/*_val.npz"))[300]   # first unseen validation puzzle
panels = parse(f.replace(".npz", ".xml"), CFG)
target = int(np.load(f)["target"])

print(os.path.basename(f), " correct answer = candidate", target)
print(f"\ncontext panel 0 holds {len(panels[0])} objects:")
for o in panels[0]:
    print("   ", o)

RAVEN_1270_val.npz  correct answer = candidate 5

context panel 0 holds 6 objects:
    {'pos': 11, 'type': 1, 'size': 2, 'color': 9, 'cx': 0.16, 'cy': 0.83}
    {'pos': 10, 'type': 0, 'size': 2, 'color': 9, 'cx': 0.83, 'cy': 0.5}
    {'pos': 7, 'type': 4, 'size': 2, 'color': 9, 'cx': 0.83, 'cy': 0.16}
    {'pos': 6, 'type': 4, 'size': 2, 'color': 9, 'cx': 0.5, 'cy': 0.16}
    {'pos': 13, 'type': 3, 'size': 2, 'color': 9, 'cx': 0.83, 'cy': 0.83}
    {'pos': 9, 'type': 1, 'size': 2, 'color': 9, 'cx': 0.5, 'cy': 0.5}


In [6]:
attrs, active = encode_puzzle(panels, CFG)
print("attrs ", tuple(attrs.shape), attrs.dtype, "  columns:", [n for n, _ in ATTRS])
print("active", tuple(active.shape), active.dtype, " ->", int(active.sum()), "real objects,",
      int((~active).sum()), "pad slots")

idx = {pos: k for k, pos in enumerate(POS_LIST[CFG])}
print(f"\npanel 0, slot by slot   (slot k = universal position {POS_LIST[CFG]}[k])")
print(f"{'slot':>4} {'univ pos':>9} {'active':>7} {'pos':>4} {'col':>4} {'shp':>4} {'siz':>4} {'pnl':>4}")
for k in range(SLOTS):
    i = 0 * SLOTS + k
    a = attrs[i].tolist()
    print(f"{k:>4} {POS_LIST[CFG][k]:>9} {str(bool(active[i])):>7} {a[0]:>4} {a[1]:>4} {a[2]:>4} {a[3]:>4} {a[4]:>4}")

attrs  (144, 5) torch.int64   columns: ['pos', 'col', 'shp', 'siz', 'pnl']
active (144,) torch.bool  -> 108 real objects, 36 pad slots

panel 0, slot by slot   (slot k = universal position [5, 6, 7, 8, 9, 10, 11, 12, 13][k])
slot  univ pos  active  pos  col  shp  siz  pnl
   0         5   False    0    0    0    0    0
   1         6    True    6    9    4    2    0
   2         7    True    7    9    4    2    0
   3         8   False    0    0    0    0    0
   4         9    True    9    9    1    2    0
   5        10    True   10    9    0    2    0
   6        11    True   11    9    1    2    0
   7        12   False    0    0    0    0    0
   8        13    True   13    9    3    2    0


The last column (`pnl`) is the panel-position embedding index: `0..7` for the eight context
panels and a single shared `8` for **every** candidate panel — a candidate is always "the
missing cell", so they must be indistinguishable by panel identity.

In [7]:
cand = 8 + target
print(f"the correct candidate (panel {cand}) holds {len(panels[cand])} objects")
print(f"{'slot':>4} {'active':>7} {'pos':>4} {'col':>4} {'shp':>4} {'siz':>4} {'pnl':>4}")
for k in range(SLOTS):
    i = cand * SLOTS + k
    a = attrs[i].tolist()
    print(f"{k:>4} {str(bool(active[i])):>7} {a[0]:>4} {a[1]:>4} {a[2]:>4} {a[3]:>4} {a[4]:>4}")

the correct candidate (panel 13) holds 6 objects
slot  active  pos  col  shp  siz  pnl
   0    True    5    3    3    2    8
   1   False    0    0    0    0    0
   2    True    7    3    1    2    8
   3    True    8    3    4    2    8
   4    True    9    3    1    2    8
   5    True   10    3    2    2    8
   6    True   11    3    2    2    8
   7   False    0    0    0    0    0
   8   False    0    0    0    0    0


## 4. What PyNeuraLogic actually receives

Two kinds of ground facts:

* **structural** (fixed for every puzzle, built once): `edge/3`, `is_context/1`, `in_cand/2`
* **feature** (rewritten per puzzle): `feat/1`, one 128-dim *weighted fact* per node, whose
  value is the torch-side embedding

Because the structure never changes, the network is grounded **once** and only the fact values
move — which is what makes training feasible at all.

In [8]:
sf = structural_facts(same_panel=False)
print(f"{len(sf)} structural facts; by predicate:")
print("   ", dict(Counter(str(x.predicate) for x in sf)))
print("\nexamples:")
for x in [sf[0], sf[len(edges(False))], sf[-1]]:
    print("   ", x)
print("\nplus one weighted fact per node, e.g.  R.feat('p0_s0')[<128 floats>]")

1800 structural facts; by predicate:
    {'edge/3': 1656, 'is_context/1': 72, 'in_cand/2': 72}

examples:
    edge(p0_s0, p1_s0, ctx_ctx).
    is_context(p0_s0).
    in_cand(p15_s8, c7).

plus one weighted fact per node, e.g.  R.feat('p0_s0')[<128 floats>]


## 5. The template

This is the whole architecture, as a logic program. Everything below is printed from the
actual `build_template()` call the run uses.

In [9]:
tmpl = build_template(level=2, d=128, layers=3, act=Transformation.LEAKY_RELU,
                      agg=Aggregation.AVG)
print(tmpl)

{128} rel_emb(same_panel).
{128} rel_emb(ctx_ctx).
{128} rel_emb(ctx_cand).
{128} rel_emb(cand_ctx).
h0(O) :- feat(O).
h0/1 [transformation=identity]
h1(Dd) :- edge(Ss, Dd, T), {128, 128} h0(Ss), rel_emb(T). [aggregation=avg]
h1(Dd) :- edge(Ss, Dd, T), {128, 128} h0(Dd). [aggregation=avg]
h1(X) :- h0(X).
h1/1 [transformation=norm]
h2(Dd) :- edge(Ss, Dd, T), {128, 128} h1(Ss), rel_emb(T). [aggregation=avg]
h2(Dd) :- edge(Ss, Dd, T), {128, 128} h1(Dd). [aggregation=avg]
h2(X) :- h1(X).
h2/1 [transformation=norm]
h3(Dd) :- edge(Ss, Dd, T), {128, 128} h2(Ss), rel_emb(T). [aggregation=avg]
h3(Dd) :- edge(Ss, Dd, T), {128, 128} h2(Dd). [aggregation=avg]
h3(X) :- h2(X).
h3/1 [transformation=norm]
candp(C) :- in_cand(O, C), {128, 128} h3(O). [aggregation=avg]
candp/1 [transformation=leakyrelu]
ctxp :- is_context(O), {128, 128} h3(O). [aggregation=avg]
ctxp/0 [transformation=leakyrelu]
sh(C) :- {128, 128} candp(C), {128, 128} ctxp.
sh/1 [transformation=leakyrelu]
{1, 128} score(C) :- sh(C).
sco

### Reading it rule by rule

**`rel_emb(<type>)`** — one learned 128-dim vector per entry of `EDGE_TYPES`. Four are declared,
but `same_panel`'s is inert in this config: no `same_panel` edges exist, so it never enters a
message. Unlike the torch GNN (a fresh `nn.Embedding` inside every layer), these are **shared
across all layers**.

**`h0(O) :- feat(O)`** with `IDENTITY` — layer 0 is just the torch embedding, passed through
unchanged.

Then, for each layer *k*, three rules whose results are **summed** into the same atom:

```
hk(D) :- edge(S,D,T), h{k-1}(S)[W1], rel_emb(T)     # message from neighbours, AVG over edges
hk(D) :- edge(S,D,T), h{k-1}(D)[W2]                 # the node's own state
hk(X) :- h{k-1}(X)                                  # identity residual, NO weight matrix
```

so the update is roughly

$$h_k(v) = \mathrm{NORM}\Big(\underbrace{\mathrm{avg}_{(u,v,t)}\big(W_1 h_{k-1}(u) + e_t\big)}_{\text{messages}} + \underbrace{W_2 h_{k-1}(v)}_{\text{self}} + \underbrace{h_{k-1}(v)}_{\text{residual}}\Big)$$

`NORM` is PyNeuraLogic's parameter-free normalisation (`transformation.join.Normalization`).
It stands in for the torch model's `LayerNorm`, which additionally learns a gain and bias.
(The self rule also carries `AVG`, but it averages the same value over every incoming edge, so
it reduces to $W_2 h_{k-1}(v)$.) Normalisation is not optional: without it, the dim-384
aligned variant diverged to a loss of ~3600.

**The head** (this is what makes it *level 2* rather than a torch head):

```
candp(C) :- in_cand(O,C), h3(O)[W]      # pool each candidate panel over its 9 slots (AVG)
ctxp     :- is_context(O), h3(O)[W]     # pool all 72 context nodes into one vector
sh(C)    :- candp(C)[W3], ctxp[W4]      # combine candidate with context
score(C) :- sh(C)[w]                    # one scalar per candidate
out      :- score(C)                    # CONCAT the 8 scalars -> the logit vector
```

`sh` summing two separately-weighted terms is algebraically the torch head's
`Linear(concat[cand, ctx])` — a linear map of a concatenation is the sum of linear maps of the
parts. What the template lacks is dropout, which has no rule-level equivalent.

## 6. The torch / PyNeuraLogic boundary

Level 2 keeps exactly two things in torch: the **attribute embeddings** (PyNeuraLogic learns
embeddings poorly — the pure-logic level 4 variant reaches only ~15%) and the **optimiser /
loss**. Message passing and scoring are the template's.

In [10]:
model = AllConstellations(level=2, embed_dim=128, hidden_dim=128, layers=3,
                          act=Transformation.LEAKY_RELU, nl_lr=1e-4,
                          agg=Aggregation.AVG, same_panel=False, layout="off")
model.eval()

torch_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
d = 128
tmpl_params = (len(EDGE_TYPES) * d            # rel_emb
               + 3 * 2 * d * d                # W1, W2 per layer
               + d * d                        # candp
               + d * d                        # ctxp
               + 2 * d * d                    # sh
               + d)                           # score
print(f"torch-side parameters : {torch_params:>8,}   ({100*torch_params/(torch_params+tmpl_params):.0f}%)")
print(f"template parameters   : {tmpl_params:>8,}   ({100*tmpl_params/(torch_params+tmpl_params):.0f}%)")
print("\ntorch-side breakdown:")
for n, p in model.named_parameters():
    if p.requires_grad and p.numel() > 0:
        print(f"   {n:<24} {tuple(p.shape)}")

torch-side parameters :   23,552   (13%)
template parameters   :  164,480   (87%)

torch-side breakdown:
   absent                   (128,)
   emb.0.weight             (22, 128)
   emb.1.weight             (10, 128)
   emb.2.weight             (5, 128)
   emb.3.weight             (6, 128)
   emb.4.weight             (9, 128)
   proj.0.weight            (128, 128)
   proj.0.bias              (128,)
   proj.1.weight            (128,)
   proj.1.bias              (128,)


### The CONCAT permutation

`Aggregation.CONCAT` emits the 8 candidate scores in the **grounder's** order, not
`c0..c7`. `AllConstellations._recover_order()` recovers it once at construction time by
running a depth-0 probe template whose node features are known markers, then inverts the
observed order into a permutation buffer applied on every forward.

In [11]:
print("recovered permutation:", model.perm.tolist())
print("(index i holds where candidate i lands in the CONCAT output)")

recovered permutation: [1, 6, 5, 0, 2, 7, 3, 4]
(index i holds where candidate i lands in the CONCAT output)


## 7. Forward pass

`forward()` is six lines: sum the five attribute embeddings, swap in the pad token for
inactive slots, project to hidden size, hand the 144 vectors to PyNeuraLogic as fact values,
reshape and un-permute.

In [12]:
import inspect
print(inspect.getsource(AllConstellations.forward))

    def forward(self, attrs, active):
        e = sum(self.emb[a](attrs[:, a]) for a in range(len(ATTRS)))
        e = torch.where(active.unsqueeze(-1), e, self.absent.expand_as(e))
        if self.layout != "off":
            occ = active.view(16, SLOTS)                           # (panels, slots)
            lay = self.emb_count(occ.sum(dim=1))                   # object count
            if self.layout == "count_occ":
                lay = lay + self.occ_proj(occ.float())             # which slots
            e = e + lay.repeat_interleave(SLOTS, dim=0)            # broadcast to nodes
        out = self.nl(self.proj(e))
        if self.level == 1:
            return self.head(out.reshape(8, self.d)[self.perm]).squeeze(-1)
        return out.reshape(8)[self.perm]



In [13]:
with torch.no_grad():
    raw = model(attrs, active)
print("untrained scores:", np.round(raw.numpy(), 3), " argmax", int(raw.argmax()), "| true", target)

untrained scores: [-0.738 -0.232 -0.655 -0.622 -0.201 -0.237 -0.495 -0.211]  argmax 4 | true 5


## 8. Load the trained weights and predict

A checkpoint is **two files** — the torch state and the template weights. `model_best.pt`
alone is not enough: `nl.internal_weights` inside it is a dummy empty Parameter, so a run that
saved only the torch side cannot be evaluated at all (an early bug here cost several runs).

The run directory is being written to by a live job, so copy the pair aside before loading
rather than reading files mid-write.

In [14]:
CKPT_DIR = "experiments/max_data_L2"        # the run this notebook documents

tmp = tempfile.mkdtemp()
for n in ("model_best.pt", "template_best.bin"):
    shutil.copy(os.path.join(CKPT_DIR, n), tmp)

ck = torch.load(os.path.join(tmp, "model_best.pt"), map_location="cpu", weights_only=False)
a = ck["args"]
assert (a["level"], a["embed_dim"], a["hidden_dim"], a["layers"]) == (2, 128, 128, 3)
assert a["no_same_panel"] and a["agg"] == "avg" and a["layout"] == "off"

model.load_state_dict(ck["torch_state"])
model.nl.model.load(os.path.join(tmp, "template_best.bin"))
model.eval()

print(f"epoch {ck['epoch']}, val {100*ck['val_acc']:.2f}%, "
      f"trained on {a['n_train']}/constellation")

epoch 78, val 68.52%, trained on 6000/constellation


In [15]:
with torch.no_grad():
    logits = model(attrs, active)
prob = torch.softmax(logits, 0)
order = torch.argsort(prob, descending=True)

print(f"{'candidate':>10} {'logit':>9} {'prob':>8}")
for c in order.tolist():
    flag = "  <-- correct" if c == target else ("  <-- predicted" if c == int(logits.argmax()) else "")
    print(f"{c:>10} {logits[c]:>9.3f} {100*prob[c]:>7.1f}%{flag}")

pred = int(logits.argmax())
print("\nanswered correctly" if pred == target else
      f"\nanswered wrongly: the correct candidate {target} is ranked "
      f"{order.tolist().index(target) + 1} of 8 ({CFG} is one of the model's weakest constellations)")

 candidate     logit     prob
         0   -20.804    29.4%  <-- predicted
         5   -20.807    29.3%  <-- correct
         7   -21.220    19.4%
         3   -21.327    17.4%
         1   -23.236     2.6%
         6   -23.916     1.3%
         2   -25.016     0.4%
         4   -25.724     0.2%

answered wrongly: the correct candidate 5 is ranked 2 of 8 (distribute_nine is one of the model's weakest constellations)


Whether this puzzle comes out right depends on the checkpoint; the last line above says. It is
a distribute_nine puzzle, one of the model's weakest constellations, so a wrong answer here is
typical rather than a fluke.

The scores share a large negative offset. That constant is meaningless: softmax and argmax depend
only on the *differences* between the eight scores.

## 9. Accuracy on unseen validation puzzles

A quick pass over a few dozen unseen validation puzzles per constellation, enough to see the
shape of the result. (`eval_pynl_test.py` scores the full test split, 2000 per constellation;
keep that for one evaluation of the final checkpoint.)

In [16]:
N = 40
rows, tot_hit, tot_n = [], 0, 0
for cfg in CONSTELLATIONS:
    hit = n = 0
    for f in sorted(glob.glob(f"data/I-RAVEN-10000/{cfg}/*_val.npz"))[300:300 + N]:
        p = parse(f.replace(".npz", ".xml"), cfg)
        at, ac = encode_puzzle(p, cfg)
        with torch.no_grad():
            hit += int(model(at, ac).argmax()) == int(np.load(f)["target"])
        n += 1
    rows.append((cfg, hit, n)); tot_hit += hit; tot_n += n

print(f"{'constellation':<40} {'correct':>9} {'acc':>8}")
for cfg, hit, n in rows:
    print(f"{cfg:<40} {hit:>4}/{n:<4} {100*hit/n:>7.1f}%")
print("-" * 59)
print(f"{'OVERALL':<40} {tot_hit:>4}/{tot_n:<4} {100*tot_hit/tot_n:>7.1f}%     (chance 12.5%)")

constellation                              correct      acc
center_single                              33/40      82.5%
distribute_four                            22/40      55.0%
distribute_nine                            15/40      37.5%
left_center_single_right_center_single     35/40      87.5%
up_center_single_down_center_single        32/40      80.0%
in_center_single_out_center_single         38/40      95.0%
in_distribute_four_out_center_single       21/40      52.5%
-----------------------------------------------------------
OVERALL                                   196/280     70.0%     (chance 12.5%)


## 10. Summary, and what was tried and rejected

```
        16 panels x 9 slots                     PyNeuraLogic template
   ┌──────────────────────────┐        ┌──────────────────────────────────────┐
   │ pos  col  shp  siz  pnl  │        │  h0 = feat                           │
   │  (5 attribute ids/node)  │        │                                      │
   └───────────┬──────────────┘        │  x3:  msg(AVG over typed edges)      │
        torch embeddings (128)         │       + self + residual -> NORM       │
        pad token for empty slots      │                                      │
               │                       │  candp(C) = AVG over slots           │
        Linear+LayerNorm+ReLU          │  ctxp     = AVG over 72 ctx nodes     │
               │                       │  score(C) = w.(W3 candp + W4 ctxp)   │
               └── 144 feat facts ────>│  out      = CONCAT score(C)           │
                                       └───────────────────┬──────────────────┘
                     8 logits <── un-permute ──────────────┘
                          │
                   CrossEntropy, AdamW (torch) + Adam on template weights
```

| tried | outcome |
|---|---|
| **6000/constellation at dim 128** (warm start from the 2000 run) | **64.00% test** at epoch 62; 63.96% from pixels; still training |
| torch GNN, both pipeline bugs fixed, 3 seeds | 45.3% ± 12.4 test (best seed 59.0%) |
| 2000/constellation (`allcons_v2`) | 53.9% val → 51.78% test; saturates ~54.4% |
| 6000/constellation at dim 256 | 27.7% val — far slower per update, 19.4 h/epoch |
| within-panel (`same_panel`) edges | hurt; dropped |
| panel layout features (count, occupancy) | hurt: 45.1% → 26.7% / 34.3% |
| level 1 (torch scoring head) | 46.2% test — the template head is better |
| level 4 (pure PyNeuraLogic, own embeddings + loss) | ~15% |

**Files:** [`train_all_constellations.py`](train_all_constellations.py) (model + training),
[`eval_pynl_test.py`](eval_pynl_test.py) (full test eval),
[`submit_max.sh`](submit_max.sh) (the chained SLURM jobs behind this run),
[`train_ladder.py`](train_ladder.py) (the level 1→4 ablation ladder).